# Deep Learning Recommenders (Part 2)

## MovieLens 100K data
In order to make direct comparison with classical methods introduced in Week 3, we will use the same MovieLens 100K data for now.

Similar to NCF, VAE does not require any additional features. We will implement an variational autoencoder to achieve the same recommendation task.

The major difference is that VAEs are commonly used for binary predictions (a.k.a. implicit feedback, which we will cover in future lectures), in the sense that the entire matrix needs to be complete (in order for predictions to be carried out). Therefore, we prepare the data slightly differently, that is, we label a rating as 1 (preferred) if the original rating value is >=4, and label it as 0 otherwise. The evaluation metrics for the binary labels will be revised accordingly.

In [1]:
# Before we start, let's first download the data and unzip it. The data is available at: https://files.grouplens.org/datasets/movielens/ml-100k.zip
# Please skip this step if you already have the data from the previous lectures.

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras import backend as K
from tensorflow.keras.losses import binary_crossentropy, BinaryCrossentropy
from tensorflow.keras.optimizers import Adam


# Load and binarize ratings
ratings = pd.read_csv('https://files.grouplens.org/datasets/movielens/ml-100k/u.data',
                      sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

# A rating is labeled as 1 if it is >= 4, otherwise it is labeled as 0
ratings['label'] = (ratings['rating'] >= 4).astype(int)

# Number of unique users and items
n_users = ratings['user_id'].nunique()
n_items = ratings['item_id'].nunique()

# Preview
print("Ratings sample:")
print(ratings.head())

# Check the shape of the datasets
print("Ratings shape:", ratings.shape) #100k ratings
# Print the number of unique users and movies
print("Number of unique users:", ratings['user_id'].nunique())
print("Number of unique movies:", ratings['item_id'].nunique())

Ratings sample:
   user_id  item_id  rating  timestamp  label
0      196      242       3  881250949      0
1      186      302       3  891717742      0
2       22      377       1  878887116      0
3      244       51       2  880606923      0
4      166      346       1  886397596      0
Ratings shape: (100000, 5)
Number of unique users: 943
Number of unique movies: 1682


## Variational Autoencoders

### Data Preparation

In [2]:
# Create user-item binary interaction matrix
interaction_matrix = np.zeros((n_users, n_items), dtype=np.float32)
for row in ratings.itertuples():
    interaction_matrix[row.user_id -1, row.item_id -1] = row.label # -1 to convert to 0-indexed

print("Interaction matrix shape:", interaction_matrix.shape)
print("Interaction matrix sample:")
print(interaction_matrix[:10, :10])  # Print a sample of the interaction matrix

# Train/test split at user level
# Recall that VAE is done at the user level. That is, it has only user latent representations but not items'.
# This is also reflected at the data splitting. That is, we split the data at user level.
# We will use 80% of the users (i.e., rows) for training and 20% for testing.
# (In contrast, recall that in the previous lectures, we split the data at the rating level, that is, 80% of the ratings, regardless of their corresponding users, are randomly assigned to the training set.)
train_matrix, test_matrix = train_test_split(interaction_matrix, test_size=0.2, random_state=42)


Interaction matrix shape: (943, 1682)
Interaction matrix sample:
[[1. 0. 1. 0. 0. 1. 1. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 1. 1. 0.]
 [0. 0. 0. 1. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 1. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 1. 0. 1. 0.]]


### Model Building

Unfortunately, there is no off-the-shelf package for VAE. We instead build our own. The code below follows Keras online example (https://keras.io/examples/generative/vae/), but with MovieLens 100K as the example. The code is also modified to avoid using Keras 3.

We first define a sampling function.

In [3]:
from tensorflow.keras import layers

class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a latent point."""

    def __init__(self, seed=1337, **kwargs):
        super().__init__(**kwargs)
        self.seed = seed

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim), seed=self.seed)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

We then define an encoder.

In [4]:
K.clear_session()

# VAE config
original_dim = n_items
intermediate_dim = 64
latent_dim = 32

# Encoder
inputs = Input(shape=(original_dim,), name='input')
h = Dense(intermediate_dim, activation='tanh')(inputs)
z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

Model: "encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input (InputLayer)          [(None, 1682)]               0         []                            
                                                                                                  
 dense (Dense)               (None, 64)                   107712    ['input[0][0]']               
                                                                                                  
 dense_1 (Dense)             (None, 32)                   2080      ['dense[0][0]']               
                                                                                                  
 dense_2 (Dense)             (None, 32)                   2080      ['dense[0][0]']               
                                                                                            

And we define a decoder.

In [5]:
# Decoder
latent_inputs = keras.Input(shape=(latent_dim,))
h_decoded = Dense(intermediate_dim, activation='tanh')(latent_inputs)
x_decoded = Dense(original_dim, activation='sigmoid')(h_decoded)
decoder = keras.Model(latent_inputs, x_decoded, name="decoder")
decoder.summary()

Model: "decoder"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32)]              0         
                                                                 
 dense_3 (Dense)             (None, 64)                2112      
                                                                 
 dense_4 (Dense)             (None, 1682)              109330    
                                                                 
Total params: 111442 (435.32 KB)
Trainable params: 111442 (435.32 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Now we are able to build our VAE. Understanding the code below, although encouraged, is beyond the scope of this course. We are only expected to run the code to get our results.

In [6]:
# VAE model
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    def call(self, inputs):
        _, _, z = self.encoder(inputs)
        return self.decoder(z)
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            # Binary crossentropy loss (mean over time steps/features, sum over batch)
            reconstruction_loss = tf.reduce_mean(
                tf.keras.losses.binary_crossentropy(data, reconstruction)
            )
            # KL divergence
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

# Build and compile the model
vae = VAE(encoder, decoder)
vae.compile(optimizer=tf.keras.optimizers.legacy.Adam())

### Training

In [7]:
# Training
# The input and output of VAE are the same, that is, both are the user-item interaction matrix.
vae.fit(train_matrix, epochs=100, batch_size=128)

Epoch 1/100
6/6 [==============================] - 0s 3ms/step - loss: 2.1637 - reconstruction_loss: 0.6949 - kl_loss: 1.3672
Epoch 2/100
6/6 [==============================] - 0s 3ms/step - loss: 1.6327 - reconstruction_loss: 0.6919 - kl_loss: 0.8943
Epoch 3/100
6/6 [==============================] - 0s 3ms/step - loss: 1.3682 - reconstruction_loss: 0.6879 - kl_loss: 0.6463
Epoch 4/100
6/6 [==============================] - 0s 3ms/step - loss: 1.1765 - reconstruction_loss: 0.6836 - kl_loss: 0.4783
Epoch 5/100
6/6 [==============================] - 0s 3ms/step - loss: 1.0591 - reconstruction_loss: 0.6804 - kl_loss: 0.3649
Epoch 6/100
6/6 [==============================] - 0s 3ms/step - loss: 0.9734 - reconstruction_loss: 0.6743 - kl_loss: 0.2880
Epoch 7/100
6/6 [==============================] - 0s 3ms/step - loss: 0.9116 - reconstruction_loss: 0.6703 - kl_loss: 0.2324
Epoch 8/100
6/6 [==============================] - 0s 3ms/step - loss: 0.8664 - reconstruction_loss: 0.6666 - kl_loss:

### Evaluation

We do not use RMSE and MAE since the ratings are binarized.

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Get model predictions
z_mean, z_log_var, z = vae.encoder(test_matrix)
reconstruction_probs = vae.decoder(z).numpy()  # shape: (n_users_test, n_items)

# Flatten true labels and predictions
y_true = test_matrix.flatten()
y_scores = reconstruction_probs.flatten()

# Threshold to get binary predictions (default: 0.5)
y_pred = (y_scores >= 0.5).astype(int)

# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

# Accuracy : 0.9658
# Precision: 0.5344
# Recall   : 0.0093
# F1 Score : 0.0182
# AUC      : 0.8656

# Which metrics is the most important for accurate recommendation? Precision or recall?

# Precision is important due to a small number of recommendation slots. 
# For example, if a user has 1000 movies in their watchlist, but only 10 slots to show on their main page, we want to ensure that the recommended movies are highly relevant. 
# In this case, precision is more important than recall.

Accuracy : 0.9657
Precision: 0.5319
Recall   : 0.0092
F1 Score : 0.0180
AUC      : 0.8664


In [9]:
# One way to improve the model is to change the threshold.
# Check the predicted probabilities first
print("Predicted probabilities sample:")
print(reconstruction_probs[:10, :10])  # Print a sample of the predicted probabilities
# Check the mean of the predicted probabilities
print("Predicted probabilities mean:", reconstruction_probs.mean())
# The mean is around 0.035, indicating that a threshold of 0.5 could be too strict

Predicted probabilities sample:
[[0.32198834 0.05046738 0.02864821 0.11536072 0.03656184 0.01417717
  0.28332907 0.14696641 0.20677917 0.04844481]
 [0.32665977 0.04971219 0.02664031 0.11768236 0.03677784 0.01313734
  0.27726838 0.14139803 0.20246486 0.04740138]
 [0.36775354 0.0691445  0.03561686 0.13952123 0.0477463  0.01899159
  0.29098862 0.17297272 0.23890425 0.06126184]
 [0.32455346 0.05624477 0.03335851 0.12677224 0.04887076 0.01828034
  0.30437243 0.17480072 0.22508699 0.05774045]
 [0.33426493 0.05434891 0.03111847 0.12604113 0.04360604 0.01679496
  0.2902754  0.15817845 0.22472498 0.04898464]
 [0.35615817 0.05287264 0.02853451 0.12607424 0.03758259 0.01431677
  0.2841307  0.16240141 0.21725054 0.04621452]
 [0.34325182 0.06761454 0.03855061 0.15534768 0.0543538  0.0212774
  0.30672744 0.17169009 0.2225528  0.06226151]
 [0.32714215 0.05720423 0.03421054 0.13091996 0.03975369 0.01650696
  0.29162756 0.15998808 0.21434022 0.04939389]
 [0.34768128 0.06277122 0.03490897 0.14966863 0.0

In [10]:
# Now we try to reduce the threshold to 0.45 (tune down slightly)
# Threshold to get binary predictions
y_pred = (y_scores >= 0.45).astype(int)

# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

# Accuracy : 0.9658
# Precision: 0.5365
# Recall   : 0.0094
# F1 Score : 0.0186
# AUC      : 0.8659

# Precision is improved, although at the cost of other measures.

# The measures are not linear with the threshold. You could try other thresholds to see if you can get a better balance.

Accuracy : 0.9657
Precision: 0.5288
Recall   : 0.0093
F1 Score : 0.0182
AUC      : 0.8664


### Generate Synthetic Users

One of the fun thing we can do with VAE is to generate synthetic users, which makes VAE a simple GenAI!

In [12]:
# Step 1: Sample a latent vector z from standard normal
latent_dim = vae.decoder.input_shape[1]  # should be 32 in your setup
z_synthetic = np.random.normal(size=(1, latent_dim)).astype(np.float32)

# Step 2: Decode to get predicted probabilities for all items
synthetic_reconstruction = vae.decoder(z_synthetic).numpy().flatten()

# Step 3: Get top-N recommendations
top_n = 10
top_items = np.argsort(synthetic_reconstruction)[-top_n:][::-1]

print("Top recommended item indices for the synthetic user:", top_items)

movie_titles = pd.read_csv(
    "https://files.grouplens.org/datasets/movielens/ml-100k/u.item",
    sep="|", encoding="latin-1", header=None, usecols=[0, 1], names=["movie_id", "title"]
)
movie_titles["item_idx"] = movie_titles["movie_id"] - 1  # 0-based for indexing

# Get titles of top items
recommended_movies = movie_titles[movie_titles["item_idx"].isin(top_items)]
print(recommended_movies.sort_values(by="item_idx"))

# The signal learned by the VAE is very strong, in the sense that the top 3 recommended movies are always the same for different synthetic users.
# This could change if we train it with a larger dataset.

Top recommended item indices for the synthetic user: [ 49  99 180  97 173 257 126   0  55 171]
     movie_id                             title  item_idx
0           1                  Toy Story (1995)         0
49         50                  Star Wars (1977)        49
55         56               Pulp Fiction (1994)        55
97         98  Silence of the Lambs, The (1991)        97
99        100                      Fargo (1996)        99
126       127             Godfather, The (1972)       126
171       172   Empire Strikes Back, The (1980)       171
173       174    Raiders of the Lost Ark (1981)       173
180       181         Return of the Jedi (1983)       180
257       258                    Contact (1997)       257
